In [17]:
# importing the necessary libraries
import requests
import pandas as pd
import json
from upsetplot import UpSet, from_contents
import matplotlib.pyplot as plt
import itertools
from functools import reduce
import mygene
import io
import time
import numpy as np
from bs4 import BeautifulSoup
import re

In [102]:
# the query protein
query = "CADM4_HUMAN"

In [103]:
# Checking if mouse or human

def check_species(query_to_check):
    split_value = query_to_check.split("_")
    if split_value[1] == "HUMAN":
        return 9606 # human tax id
    if split_value[1] == "MOUSE":
        return 10090 # mouse tax id
   
# test 
species_tax_id = check_species(query)
print(species_tax_id)
    

9606


In [19]:
# Function that takes a ppi dataframe in the format column1-interactor-of-query & column2-query, and outputs from those two columns only one column with the interactor of the query (target protein)

def get_interactors_for_target(df, column_a, column_b, target_protein):
    """
    Takes a ppi dataframe in the format column1:interactor-of-query & column2:query, 
    and outputs from those two columns only one column with the interactor of the query (target protein)
    
    Args:
        df: the input dataframe.
        column_a: a column that holds either the query of the interactor of the query.
        column_b: a column that holds either the query of the interactor of the query.
        target_protein: the query where you want to find the interactors of.
    
    Returns:
        df: dataframe with one column less then input dataframe but now with only the interactor
        of the query.
    """    
    def get_interactors(row):
        if row[column_a] == target_protein:
            return row[column_b]
        elif row[column_b] == target_protein:
            return row[column_a]
        else:
            return None
    df["interactor_of_" + target_protein] = df.apply(get_interactors, axis = 1)
    
    return df

In [4]:
def convert_uniprotID_uniprotAcNr(uniprotID_or_AcNr): # e.g. can be P19320 or VCAM1_HUMAN
    """
    Converts a uniprotID (eg VCAM1_HUMAN) to a uniprot Accession Number (eg P19320), or 
    the other way around. This is done one by one, thus not in batch retrieval.
    """

    uniprot_api_url = "https://rest.uniprot.org/uniprotkb" 
    format = "json"
    uniprot_request_url = f"{uniprot_api_url}/{uniprotID_or_AcNr}?format={format}"
    
    uniprot_response = requests.get(uniprot_request_url)
    
    if uniprot_response.ok:       
        uniprot_dict = uniprot_response.json() 

        if "_" in uniprotID_or_AcNr:
            value = uniprot_dict["primaryAccession"]
        else:
            value = uniprot_dict["uniProtkbId"]
        return (value)
    
    else: 
        print("Data retrieval through Uniprot API failed, for the function convert_uniprotID_uniprotAcNr")

# test
id_or_name = convert_uniprotID_uniprotAcNr("Q61490")
print(id_or_name)

CD166_MOUSE


In [20]:
# this function extracts only the main protein & not the isoforms from a list of uniprot isoform proteins, which is used in the next function (convert_uniprotID_uniprotAcNr_from_df_column)

def extract_main_protein(isoform_list):
    """
    Extracts only the main proteins from a isoform list of uniprot accession numbers.
    This function is used in the function "convert_uniprotID_uniprotAcNr_from_df_column".
    """
    convert_None_to_NaN = [np.nan if isoform is None else isoform for isoform in isoform_list] # convert None to NaN
    convert_float_to_str = [str(isoform) if isoform is np.nan else isoform for isoform in convert_None_to_NaN] # conversion of NaN floats to NaN strings
    protein_list = list(filter(lambda isoform: "-" not in isoform, convert_float_to_str)) # the iteration, filtering out the isoforms
    
    return protein_list

# test
input_list = ["L1CAM_MOUSE", "L1CAM_HUMAN", "CALM_DROME", None]
input_list2 = ["P29533-1", "P29533", "Q28260", "Q13349"]

test = extract_main_protein(input_list2)
print(test)

['P29533', 'Q28260', 'Q13349']


In [21]:
# ID mapping from the Uniprot-API is used to batch retrieve uniprotIDs or uniprotNames

def convert_uniprotID_uniprotAcNr_from_df_column(df, column_to_convert, new_column_name, batch_size = 10):
    """     
    Converts a pandas dataframe column that contains uniprot accession numbers (e.g. Q13740) 
    to uniprot IDs (e.g. CD166_HUMAN) & vice versa. It uses the Uniprot-API ID mapping
    and does this in batch retrieval so it has one large payload for in POST request.
    
    Args:
        df: the dataframe that contains the column that needs to be converted.
        columnd_to_convert: the column in the dataframe that needs conversion, 
            this column holds either uniprot IDs or accession numbers.
        new_column_name: name of the new column where the converted items 
            will be stored.
        batch_size: the size of batches for the API request, this is needed because the 
            returned JSON is too big. 
    
    Returns:
        An exact copy of the original dataframe but it holds an extra column that 
        contains the converted items, either uniprot IDs or names. It will also print
        the job ID so that you can check with Postman for example if it is not returning
        the expected output.
    
    Raises:
        JOB status error: job status is not running & will not get results, thus something 
            went wrong with job
        ValueError: The returned list from UniProt-API is not equal in length to the nr of 
            rows in the original dataframe
    """
    
    # Conversions for input into Uniprot-API
    df[column_to_convert] = df[column_to_convert].astype(str) # making sure it is a string
    list_of_uniprotIDs_or_Names = df[column_to_convert].tolist() # column to list
    
    # Split the list into batches
    batches = [list_of_uniprotIDs_or_Names[i:i + batch_size] for i in range(0, len(list_of_uniprotIDs_or_Names), batch_size)]

    # making a list to append the results
    desired_id_proteins =[]
    
    # batch process
    for batch in batches: 
        ls = ",".join(batch) # conversion to input as payload to uniprot
        
        # making the POST request
        r = requests.post("https://rest.uniprot.org/idmapping/run", data={
            "from": "UniProtKB_AC-ID",
            "to": "UniProtKB", 
            "ids": ls})

        # getting the job ID nr
        job_id = r.json()["jobId"]
        print("job ID for ID mapping through Uniprot:", job_id)

        # for loop that checks whether the job is running and when job is finished it will return a list
        while True:
            response = requests.get(f"https://rest.uniprot.org/idmapping/status/{job_id}")
            data_json = json.loads(response.text)
        
            if "jobStatus" in data_json:
                job_status = data_json["jobStatus"]
                print(f"job status: {job_status}")
                
                if job_status != "RUNNING":
                    print("JobError: job status is not running, error with posting the request")
                    break
        
            if "results" in data_json:
                if "_" in ls:
                    for entry in batch:
                        matched = False
                        for json_entry in data_json["results"]:
                            if entry == json_entry["from"]:
                                desired_id_proteins.append(json_entry["to"]["primaryAccession"])
                                matched = True
                                break
                        if not matched:
                            desired_id_proteins.append("nan")   
                            
                else:
                    for entry in batch:
                        matched = False
                        for json_entry in data_json["results"]:
                            if entry == json_entry["from"]:  
                                desired_id_proteins.append(json_entry["to"]["uniProtkbId"])
                                matched = True
                                break
                        if not matched:
                            desired_id_proteins.append("nan")
                
                break
            
            time.sleep(5) # wait for 5sec before the looop begins again
        
    # check before appending list to original dataframe, if the list has the same nr of items as the original df has rows
    if len(desired_id_proteins) != len(list_of_uniprotIDs_or_Names):
        raise ValueError("Data appending conflict: the returned list from UniProt-API is not equal in length to the nr of rows in the original dataframe")

    # now append list to dataframe   
    df[new_column_name] = desired_id_proteins
    
    return df

# testing the function
# dummy dataframe
dummy_data = {"Name":["Karan","Rohit","Sahil","Aryan"],"Protein":["P29533-1", "P29533", "Q28260", "Q13349"]}
dummy_df = pd.DataFrame(dummy_data)

# testing
df_test = convert_uniprotID_uniprotAcNr_from_df_column(dummy_df, "Protein",  "UniprotName_B")
df_test.head()

job ID for ID mapping through Uniprot: f2a543c34ef6a1eded37f569b44db4634abd9a0a


,Name,Protein,UniprotName_B
0,Karan,P29533-1,VCAM1_MOUSE
1,Rohit,P29533,VCAM1_MOUSE
2,Sahil,Q28260,VCAM1_CANLF
3,Aryan,Q13349,ITAD_HUMAN


In [ ]:
# ## Backup function

# import requests
# import json
# import time

# def extract_main_protein(desired_id):
#     # Assuming you have the implementation for this function
#     pass

# def convert_uniprotID_uniprotAcNr_from_df_column(df, column_to_convert, new_column_name, batch_size=10):
#     """
#     Converts a pandas dataframe column that contains UniProt accession numbers to UniProt IDs and vice versa.
#     Uses the UniProt API ID mapping and retrieves results in batches.

#     Args:
#         df: the dataframe that contains the column that needs to be converted.
#         column_to_convert: the column in the dataframe that needs conversion,
#             this column holds either UniProt IDs or accession numbers.
#         new_column_name: name of the new column where the converted items
#             will be stored.
#         batch_size: size of each batch for API requests.

#     Returns:
#         An updated dataframe with an extra column that contains the converted items.

#     Raises:
#         JOB status error: job status is not running & will not get results, thus something
#             went wrong with the job.
#         ValueError: The returned list from UniProt-API is not equal in length to the number of
#             rows in the original dataframe.
#     """

#     # Conversions for input into UniProt-API
#     df[column_to_convert] = df[column_to_convert].astype(str)  # making sure it is a string
#     list_of_uniprotIDs_or_Names = df[column_to_convert].tolist()  # column to list

#     # Split the list into batches
#     batches = [list_of_uniprotIDs_or_Names[i:i + batch_size] for i in range(0, len(list_of_uniprotIDs_or_Names), batch_size)]

#     # List to store the results
#     desired_id_proteins = []

#     # Process batches
#     for batch in batches:
#         ls = ",".join(batch)  # conversion to input as payload to UniProt
#         print(ls)

#         # Making the POST request
#         r = requests.post("https://rest.uniprot.org/idmapping/run", data={
#             "from": "UniProtKB_AC-ID",
#             "to": "UniProtKB",
#             "ids": ls
#         })

#         # Getting the job ID nr
#         job_id = r.json()["jobId"]
#         print("job ID for ID mapping through UniProt:", job_id)

#         # For loop that checks whether the job is running and waits for completion
#         while True:
#             response = requests.get(f"https://rest.uniprot.org/idmapping/status/{job_id}")
#             data_json = json.loads(response.text)

#             if "jobStatus" in data_json:
#                 job_status = data_json["jobStatus"]
#                 print(f"job status: {job_status}")

#                 if job_status != "RUNNING":
#                     print("JobError: job status is not running, error with posting the request")
#                     break

#             if "results" in data_json:
#                 if "_" in ls:
#                     for entry in batch:
#                         matched = False
#                         for json_entry in data_json["results"]:
#                             if entry == json_entry["from"]:
#                                 desired_id_proteins.append(json_entry["to"]["primaryAccession"])
#                                 matched = True
#                                 break
#                         if not matched:
#                             desired_id_proteins.append("nan")

#                 else:
#                     for entry in batch:
#                         matched = False
#                         for json_entry in data_json["results"]:
#                             if entry == json_entry["from"]:
#                                 desired_id_proteins.append(json_entry["to"]["uniProtkbId"])
#                                 matched = True
#                                 break
#                         if not matched:
#                             desired_id_proteins.append("nan")

#                 break

#             time.sleep(5)  # wait for 5 sec before the loop begins again

#     # Check before appending list to the original dataframe if the list has the same number of items as the original df has rows
#     if len(desired_id_proteins) != len(list_of_uniprotIDs_or_Names):
#         raise ValueError("Data appending conflict: the returned list from UniProt-API is not equal in length to the number of rows in the original dataframe")

#     # Now append list to dataframe
#     df[new_column_name] = desired_id_proteins

#     return df

# df_test = convert_uniprotID_uniprotAcNr_from_df_column(df_biogrid_test, "UniprotID_A",  "UniprotName_A")
# df_test.head()


In [22]:
# Function that uses the MyGene-API to convert the entrezGeneID to UniprotID (this is for biogrid, because biogrid outputs only gene names), 
# this function is only used in the development stage

def convert_geneID_uniprotID(geneID):

    mygene_api_url = "https://mygene.info/v3/gene"
    entrezGeneID = geneID
    mygene_request_url = f"{mygene_api_url}/{entrezGeneID}?fields=all&dotfield=false&size=10"

    mygene_response = requests.get(mygene_request_url)
    
    if mygene_response.ok:     
        mygene_json = mygene_response.json()

        if "uniprot" in mygene_json and "Swiss-Prot" in mygene_json["uniprot"]:
            uniprot_id = mygene_json["uniprot"]["Swiss-Prot"]
        elif "pantherdb" in mygene_json and "uniprot_kb" in mygene_json["pantherdb"]:
            uniprot_id = mygene_json["pantherdb"]["uniprot_kb"]
        else:
            uniprot_id = "NA"
            
        return uniprot_id
    
    else:
        print("Data retrieval through MyGene API failed, for the function convert_geneID_uniprotID")

# example
Uniprot_convert = convert_geneID_uniprotID(6047)
print(Uniprot_convert)

P78317


In [23]:
# Function that uses the MyGene-API python package to convert entrezGeneIDs to UniprotIDs in batch retrieval (this is for biogrid)
# This is an alternative to the function above, this uses the MyGene package instead querying the each input seperately through the API

def convert_geneIDs_uniprotAcNr(df, df_column_name, df_new_column_name):
    """
    Using the MyGene-API python package this function converts entrez gene IDs to uniprot ID.
    It takes as input a dataframe column of which it will retrieve the uniprot accession numbers
    in batch retrieval and then append this list to the original dataframe.
    
    Args:
        df: original dataframe
        df_column_name: name of the column of which the entrez gene ID data will be send as an endpoint.
        df_new_column_name: name of the new column with the Uniprot accession number data.
        
    Returns:
        df: original dataframe with a new column of uniprot accession numbers.
    """

    mg = mygene.MyGeneInfo()
    
    # convert column to list
    list_of_column = df[df_column_name].tolist()
    
    # get the data from MyGene-API package
    df_mg = mg.getgenes(list_of_column, fields = "uniprot", as_dataframe = True)
    
    # filter it, convert to list & add column to original df
    df_mg_filtered = df_mg[["uniprot.Swiss-Prot"]]
    list_filtered = df_mg_filtered["uniprot.Swiss-Prot"].tolist()
    
    # append the list to the dataframe
    df[df_new_column_name] = list_filtered
    
    return df

In [63]:
# Function that uses the UniProt API to get the protein Name from the string ID (this is for string db)

def convert_stringID_to_uniprotName(string_id):
    """
    Converts string gene ID to uniprot ID using the uniprot-API,
    not in batch retrieval.
    """
    uniprot_api_url = "https://rest.uniprot.org/uniprotkb/search?query=gene_exact:"
    uniprot_request_url = f"{uniprot_api_url}{string_id}+AND+organism_id:{species_tax_id}"

    uniprot_response = requests.get(uniprot_request_url)
    
    if uniprot_response.ok:        
        uniprot_json = uniprot_response.json()

        if "results" in uniprot_json and len(uniprot_json["results"]) > 0:
            uniprot_name = uniprot_json["results"][0]["uniProtkbId"]
        else:
            uniprot_name = None

        return uniprot_name
    
    else:
        print("Data retrieval through Uniprot API failed, for the function convert_stringID_to_uniprotName")


# example
l = convert_stringID_to_uniprotName("HAVCR1")
print(l)


HAVR1_MOUSE


In [25]:
# If you have a df with duplicate values for a certain column but not duplicate value for the other columns, this function will make of these rows, one row but retaining the info

def removeDuplicateRow_butRetainInfo (df, columnWithDuplicate, columnRetain1, columnRetain2, columnRetain3, columnRetain4 = None):
    """
    This function will take a dataframe that has duplicate rows for a certain column,
    which in this case is the interactor of the query. It will take the data from 
    certain specified columns and merge them into one row and retaining the info from the columns.
    
    Args:
        df: input dataframe
        columnWithDuplicate: the column which can hold duplicate values
        columnRetain1: column from the data should be retained
        columnRetain2: column from the data should be retained
        columnRetain3: column from the data should be retained
        columnRetain4: column from the data should be retained, this column is not necessary
    
    Returns:
        A dataframe that has no duplicate rows and contains all the info from the original dataframe.
    """
    
    def join_columnValues(series):
        return ", ".join(str(value) for value in series)

    columns_to_aggregate = {
        columnRetain1: join_columnValues,
        columnRetain2: join_columnValues,
        columnRetain3: join_columnValues}
    
    if columnRetain4 is not None:
        columns_to_aggregate[columnRetain4] = join_columnValues
        
    df_final = df.groupby(columnWithDuplicate).agg(columns_to_aggregate).reset_index()
    
    return df_final

In [127]:
# converting the from human uniprotAcNr to mouseMGI using the panther API, not in batch retrieval

def convert_uniprotAcNr_to_mouseMGI(uniprotAcNr, species):
    """
    Converts a uniprot accession number to mouse MGI ID using the panther-API,
    not in batch retrieval.
    """
    panther_api_url = "https://pantherdb.org/services/oai/pantherdb/ortholog/matchortho?"
    originOrganism = species
    targetOrganism = "10090"
    panther_request_url = f"{panther_api_url}geneInputList={uniprotAcNr}&organism={originOrganism}&targetOrganism={targetOrganism}&orthologType=all"
    
    panther_response = requests.post(panther_request_url)
    
    if panther_response.ok:      
        panther_json = panther_response.json()
        
        mgi_ids = []
        if "search" in panther_json and "mapping" in panther_json["search"]:
            mapping = panther_json["search"]["mapping"]
            
            if "mapped" in mapping:
                if isinstance(mapping["mapped"], list):
                    for mapped_gene in mapping["mapped"]:
                        try:
                            if species_tax_id == 10090:
                                mgi_id = mapped_gene["target_gene"].split("|")[1].split("=")[1]
                            elif species_tax_id == 9606:
                                mgi_id = mapped_gene["gene"].split("|")[1].split("=")[1]  
                        except IndexError:
                            mgi_id = "NA"
                        mgi_ids.append(mgi_id)
                # elif isinstance(mapping["mapped"], dict):
                #     try:
                #         if species_tax_id == 10090:
                #             mgi_id = mapping["mapped"]["target_gene"].split("|")[1].split("=")[1]
                #         elif species_tax_id == 9606:
                #             mgi_id = mapping["gene"].split("|")[1].split("=")[1] 
                #     except (KeyError, IndexError):
                #         mgi_id = "NA"
                #     mgi_ids.append(mgi_id)
        
        return mgi_ids
    
    else:
        print("Data retrieval through Panther API failed, for the function: convert_uniprotAcNr_to_mouseMGI")
   
# test
species_tax_id = 9606
test = convert_uniprotAcNr_to_mouseMGI("Q00005", species_tax_id)
print(test)


[]


In [123]:
import requests

def convert_uniprotAcNr_to_mouseMGI(uniprotAcNr, species_tax_id):
    panther_api_url = "https://pantherdb.org/services/oai/pantherdb/ortholog/matchortho?"
    originOrganism = species_tax_id
    targetOrganism = "10090"
    panther_request_url = f"{panther_api_url}geneInputList={uniprotAcNr}&organism={originOrganism}&targetOrganism={targetOrganism}&orthologType=all"
    
    panther_response = requests.post(panther_request_url)
    
    if panther_response.ok:      
        panther_json = panther_response.json()
        
        mgi_ids = []
        if "search" in panther_json and "mapping" in panther_json["search"]:
            mapping = panther_json["search"]["mapping"]
            
            if "mapped" in mapping:
                mapped_gene = mapping["mapped"]
                try:
                    if species_tax_id == 10090:
                        mgi_id = mapped_gene["target_gene"].split("|")[1].split("=")[1]
                    elif species_tax_id == 9606:
                        mgi_id = mapped_gene["gene"].split("|")[1].split("=")[1]  
                except (KeyError, IndexError):
                    mgi_id = "NA"
                mgi_ids.append(mgi_id)
        
        return mgi_ids
    
    else:
        print("Data retrieval through Panther API failed, for the function: convert_uniprotAcNr_to_mouseMGI")
        return []

# test
species_tax_id = 9606
test = convert_uniprotAcNr_to_mouseMGI("Q00005", species_tax_id)
print(test)


['9305']


In [27]:
# make a function for putting the data into the right format for the upsetplot (at the end of the script)

def convert_column_to_list(df, column):
    """
    Converts a column to a flattened list, specifcally for the upsetplot.
    """
    column_list = df[[column]].values.tolist()
    
    def flatten_list(nested_list):
        return list(itertools.chain(*nested_list))

    interactors = flatten_list(column_list)

    return interactors

In [88]:
# Conversion to uniprot accession number, because some 3 out of the 4 database likes uniprot accession numbers
query_AcNr = convert_uniprotID_uniprotAcNr(query)

In [86]:
# Extracting the data from the BioGRID API

# BioGRID personal Access Key: caf14dfd9a0b7be447d282c322b8362e

biogrid_api_url = "https://webservice.thebiogrid.org/interactions"

geneList = [convert_uniprotID_uniprotAcNr(query)]

print(geneList)

params = {
    "accesskey": "caf14dfd9a0b7be447d282c322b8362e", # need to request
    "additionalIdentifierTypes": "UNIPROT",
    "format": "json",
    "geneList": geneList,
    "taxId": species_tax_id, 
    "max": 100000
}

response = requests.get(biogrid_api_url, params=params)

if response.ok:
    biogrid_interactions = response.json()

    print(biogrid_interactions)
    
    if not biogrid_interactions:
        biogrid_data = {}
    
    else:
        biogrid_data = {}
        for interaction_id, interaction in biogrid_interactions.items():
            biogrid_data[interaction_id] = interaction
            biogrid_data[interaction_id]["INTERACTION_ID"] = interaction_id
else:
    print("Access to BioGrid database failed")
    
# loading into dataframe
biogrid_df = pd.DataFrame.from_dict(biogrid_data, orient="index")

columns = [
    "INTERACTION_ID",
    "ENTREZ_GENE_A",
    "ENTREZ_GENE_B",
    "OFFICIAL_SYMBOL_A",
    "OFFICIAL_SYMBOL_B",
    "EXPERIMENTAL_SYSTEM",
    "PUBMED_ID",
    "PUBMED_AUTHOR",
    "THROUGHPUT",
    "QUALIFICATIONS"]

biogrid_df = biogrid_df[columns]

biogrid_df.head(5)

['P29533']
{'2341296': {'BIOGRID_INTERACTION_ID': 2341296, 'ENTREZ_GENE_A': '211651', 'ENTREZ_GENE_B': '22329', 'BIOGRID_ID_A': 229254, 'BIOGRID_ID_B': 204505, 'SYSTEMATIC_NAME_A': '-', 'SYSTEMATIC_NAME_B': '-', 'OFFICIAL_SYMBOL_A': 'Fancd2', 'OFFICIAL_SYMBOL_B': 'Vcam1', 'SYNONYMS_A': '2410150O07Rik|AU015151|BB137857|FA-D2|FA4|FACD|FAD|FANCD', 'SYNONYMS_B': 'CD106|Vcam-1', 'EXPERIMENTAL_SYSTEM': 'Affinity Capture-MS', 'EXPERIMENTAL_SYSTEM_TYPE': 'physical', 'PUBMED_AUTHOR': 'Zhang T (2017)', 'PUBMED_ID': 28378742, 'ORGANISM_A': 10090, 'ORGANISM_B': 10090, 'THROUGHPUT': 'High Throughput', 'QUANTITATION': '-', 'MODIFICATION': '-', 'ONTOLOGY_TERMS': {}, 'QUALIFICATIONS': '-', 'TAGS': '-', 'SOURCEDB': 'BIOGRID'}, '2688021': {'BIOGRID_INTERACTION_ID': 2688021, 'ENTREZ_GENE_A': '668218', 'ENTREZ_GENE_B': '22329', 'BIOGRID_ID_A': 580016, 'BIOGRID_ID_B': 204505, 'SYSTEMATIC_NAME_A': '-', 'SYSTEMATIC_NAME_B': '-', 'OFFICIAL_SYMBOL_A': 'Bin2', 'OFFICIAL_SYMBOL_B': 'Vcam1', 'SYNONYMS_A': '-', 'S

,INTERACTION_ID,ENTREZ_GENE_A,ENTREZ_GENE_B,OFFICIAL_SYMBOL_A,OFFICIAL_SYMBOL_B,EXPERIMENTAL_SYSTEM,PUBMED_ID,PUBMED_AUTHOR,THROUGHPUT,QUALIFICATIONS
2341296,2341296,211651,22329,Fancd2,Vcam1,Affinity Capture-MS,28378742,Zhang T (2017),High Throughput,-
2688021,2688021,668218,22329,Bin2,Vcam1,Co-fractionation,32325033,Pourhaghighi R (2020),High Throughput,High confidence interactions had an EPIC Score...
2688022,2688022,14218,22329,Sh3pxd2a,Vcam1,Co-fractionation,32325033,Pourhaghighi R (2020),High Throughput,High confidence interactions had an EPIC Score...


In [16]:
# dataframe operations

df_biogrid = (biogrid_df
              .assign(**{"UniprotID_A": lambda x: x["ENTREZ_GENE_A"].map(convert_geneID_uniprotID)})
              .assign(**{"UniprotID_B": lambda x: x["ENTREZ_GENE_B"].map(convert_geneID_uniprotID)}) # converting entrez gene IDs to uniprot AcNr
              .pipe(convert_uniprotID_uniprotAcNr_from_df_column, "UniprotID_A", "UniprotName_A") # converting uniprotID to uniprot AcNr
              .pipe(convert_uniprotID_uniprotAcNr_from_df_column, "UniprotID_B", "UniprotName_B") # converting uniprotID to uniprot AcNr
              .filter(items = { "UniprotName_A", 
                                "UniprotName_B", 
                                "EXPERIMENTAL_SYSTEM", 
                                "PUBMED_ID",
                                "PUBMED_AUTHOR"}) # filtering the columns
              .rename(columns = {       "UniprotName_A": "biogrid_interactor_a", 
                                        "UniprotName_B": "biogrid_interactor_b",
                                        "EXPERIMENTAL_SYSTEM": "biogrid_method",
                                        "PUBMED_ID": "biogrid_pubID",
                                        "PUBMED_AUTHOR": "biogrid_publication"}) # renaming the columns
              .pipe(get_interactors_for_target, "biogrid_interactor_a", "biogrid_interactor_b", query) # only retaining the interactors column
              .pipe(removeDuplicateRow_butRetainInfo, "interactor_of_" + query ,"biogrid_publication", "biogrid_method", "biogrid_pubID") # removing duplicate rows but retainig info
              .assign(**{"interactor_of_" + query: lambda x: x["interactor_of_" + query].replace("nan", np.nan)}) # replacing the nan values with NaN 
              .dropna(subset = ["interactor_of_" + query])) # dropping all NaN values in the interactor column


df_biogrid.head(5)

job ID for ID mapping through Uniprot: c3b49517ed368fbcb75dbf757bacaa4023874c2a
job status: RUNNING
job ID for ID mapping through Uniprot: 9e4792ac222e003639e785fc383df343593e0c10
job status: RUNNING
job ID for ID mapping through Uniprot: 28ae837c827bef3aff4641fd35a70b4c1925c530
job status: RUNNING
job ID for ID mapping through Uniprot: d3d81389cd9814fa3ed58ef998d32a0cff827aab
job status: RUNNING
job ID for ID mapping through Uniprot: 6f502db935ec82c3e1501ba61557e3e91819d1e7
job status: RUNNING
job ID for ID mapping through Uniprot: 5844d12844f6b3559b5eeba1113bc851257a1417
job status: RUNNING
job ID for ID mapping through Uniprot: 6f502db935ec82c3e1501ba61557e3e91819d1e7
job ID for ID mapping through Uniprot: 6f502db935ec82c3e1501ba61557e3e91819d1e7
job ID for ID mapping through Uniprot: 6f502db935ec82c3e1501ba61557e3e91819d1e7
job ID for ID mapping through Uniprot: af4e54013485bfdc7e7c61326000b551c435b358
job status: RUNNING
job ID for ID mapping through Uniprot: a099ac3a112d9544474ea

,interactor_of_G37L1_HUMAN,biogrid_publication,biogrid_method,biogrid_pubID
0,AGRB3_HUMAN,Luck K (2020),Two-hybrid,32296183
1,AQP10_HUMAN,Luck K (2020),Two-hybrid,32296183
2,AQP1_HUMAN,Luck K (2020),Two-hybrid,32296183
3,AR13B_HUMAN,Luck K (2020),Two-hybrid,32296183
4,CISD2_HUMAN,Luck K (2020),Two-hybrid,32296183


In [89]:
# calling the API
intact_api_url = "http://www.ebi.ac.uk/Tools/webservices/psicquic/intact/webservices"

version = "current"
method = "interactor"
protein = query_AcNr
format = "tab25"

intact_request_url = f"{intact_api_url}/{version}/search/{method}/{protein}?format={format}"

intact_response = requests.get(intact_request_url)

if intact_response.ok:
    int_act_interactions = intact_response.text
else:
    print("Access to IntAct database failed")
    
# Loading into a dataframe
intact_columns = [  "Unique identifier for interactor A", 
                    "Unique identifier for interactor B", 
                    "Alternative identifier for interactor A", 
                    "Alternative identifier for interactor B", 
                    "Aliases for A", 
                    "Aliases for B", 
                    "Interaction detection methods", 
                    "First author", 
                    "Identifier of the publication", 
                    "NCBI Taxonomy identifier for interactor A", 
                    "NCBI Taxonomy identifier for interactor B", 
                    "Interaction types", 
                    "Source databases", 
                    "Interaction identifier(s)", 
                    "Confidence score"]

if not int_act_interactions: # checking if the response is empty
    intact_df = pd.DataFrame(columns = intact_columns)
else:
    intact_rows = [row.split("\t") for row in int_act_interactions.split("\n")]
    intact_df = pd.DataFrame(intact_rows, columns = intact_columns)


In [ ]:
intact_df_pipe = (intact_df
                .filter(items = { "Unique identifier for interactor A", 
                                            "Unique identifier for interactor B", 
                                            "Interaction detection methods", 
                                            "First author", 
                                            "Identifier of the publication", 
                                            "Confidence score"}) # filter columns
                .iloc[:-1] # remove last row, it holds no data when retrieving it through the API
                .assign(**{"Unique identifier for interactor A": lambda x: x["Unique identifier for interactor A"].str.removeprefix("uniprotkb:"),
                            "Unique identifier for interactor B": lambda x: x["Unique identifier for interactor B"].str.removeprefix("uniprotkb:")}) # removing the uniprotkbID refix from the name
                .loc[~intact_df["Unique identifier for interactor B"].str.contains("intact:", na = False)] # filtering out rows with IntactID instead of UniprotID
                .pipe(convert_uniprotID_uniprotAcNr_from_df_column, "Unique identifier for interactor A", "UniprotName_A") # converting uniprot ID to uniprot Ac Nr
                .pipe(convert_uniprotID_uniprotAcNr_from_df_column, "Unique identifier for interactor B", "UniprotName_B") # converting uniprot ID to uniprot Ac Nr
                .drop_duplicates() # dropping duplicates
                .rename(columns= {  "UniprotName_A": "IntAct_interactor_a", 
                                    "UniprotName_B": "IntAct_interactor_b",
                                    "Interaction detection methods": "IntAct_method",
                                    "First author": "IntAct_publication",
                                    "Identifier of the publication": "IntAct_pubID",
                                    "Confidence score": "IntAct_score"}) # renaming columns
                .pipe(get_interactors_for_target, "IntAct_interactor_a", "IntAct_interactor_b", query) # get only the interactors column
                .assign(**{"IntAct_score": lambda x: x["IntAct_score"].str.removeprefix("intact-miscore:")}) # removing a prefix from a certain column value
                .pipe(removeDuplicateRow_butRetainInfo, "interactor_of_" + query, "IntAct_publication", "IntAct_method", "IntAct_pubID", "IntAct_score") ) 

intact_df_pipe.shape
intact_df_pipe.head(5)

In [ ]:
# cleaning up the dataframe

# filter out the columns that are needed
intact_df_filter = intact_df[["Unique identifier for interactor A", 
                                  "Unique identifier for interactor B", 
                                  "Interaction detection methods", 
                                  "First author", 
                                  "Identifier of the publication", 
                                  "Confidence score"]]

# remove the last row (this is a row with no information, an empty row)
intact_df_filter = intact_df_filter[:-1]

# removing the uniprotkbID refix from the name
intact_df_filter[["Unique identifier for interactor A", "Unique identifier for interactor B"]] = intact_df_filter[["Unique identifier for interactor A", "Unique identifier for interactor B"]].map(lambda x: x.removeprefix("uniprotkb:"))

# filtering out rows with IntactID instead of UniprotID
intact_df_filter = intact_df_filter[~intact_df_filter["Unique identifier for interactor B"].str.contains("intact:")]

# converting the uniprot accession numbers to uniprot IDs
intact_df_filter2 = convert_uniprotID_uniprotAcNr_from_df_column(intact_df_filter, "Unique identifier for interactor A", "UniprotName_A")
intact_df_filter3 = convert_uniprotID_uniprotAcNr_from_df_column(intact_df_filter2, "Unique identifier for interactor B", "UniprotName_B")

intact_df_filter3.head(5)
intact_df_filter3.shape

In [ ]:
# In the interest of time, I splitted the above cell up

# removing exact duplicate rows
df_intact_dupl = intact_df_filter3.drop_duplicates()


# rename the column headers, with prefix IntAct
df_intAct_final = df_intact_dupl.rename(columns= {"UniprotName_A": "IntAct_interactor_a", 
                                                    "UniprotName_B": "IntAct_interactor_b",
                                                    "Interaction detection methods": "IntAct_method",
                                                    "First author": "IntAct_publication",
                                                    "Identifier of the publication": "IntAct_pubID",
                                                    "Confidence score": "IntAct_score"})

# get one column, only the interactor of the query
df_IntAct = get_interactors_for_target(df_intAct_final, "IntAct_interactor_a", "IntAct_interactor_b", query)

# clean up the IntAct_score column, that it has only the score and the string ("intact-miscore:")
df_IntAct[["IntAct_score"]] = df_IntAct[["IntAct_score"]].map(lambda x: x.removeprefix("intact-miscore:"))

# Now we have some duplicate rows based on the "interactor_of_query" column, so we joining duplicate rows but retaining the information, so converting all the info to one row
df_IntAct = removeDuplicateRow_butRetainInfo(df_IntAct, "interactor_of_" + query, "IntAct_publication", "IntAct_method", "IntAct_pubID", "IntAct_score")

df_IntAct.head(5)
df_IntAct.shape

In [111]:
# Extracting the data from the STRING-API

string_api_url = "https://string-db.org//api"

output_format = "json"
method = "interaction_partners"

string_request_url = "/".join([string_api_url, output_format, method])

params = {
    "identifiers": query,
    "species": species_tax_id, # human
    "required_score": 400,
    "limit": 1000000 
}

response = requests.post(string_request_url, data = params)

if response.ok:
    string_raw_data = response.text

print(string_raw_data)

[{"stringId_A": "9606.ENSP00000222374", "stringId_B": "9606.ENSP00000222644", "preferredName_A": "CADM4", "preferredName_B": "MPP6", "ncbiTaxonId": 9606, "score": 0.888, "nscore": 0, "fscore": 0, "pscore": 0, "ascore": 0, "escore": 0.045, "dscore": 0, "tscore": 0.888}, {"stringId_A": "9606.ENSP00000222374", "stringId_B": "9606.ENSP00000357106", "preferredName_A": "CADM4", "preferredName_B": "CADM3", "ncbiTaxonId": 9606, "score": 0.862, "nscore": 0, "fscore": 0, "pscore": 0, "ascore": 0.128, "escore": 0, "dscore": 0, "tscore": 0.849}, {"stringId_A": "9606.ENSP00000222374", "stringId_B": "9606.ENSP00000357110", "preferredName_A": "CADM4", "preferredName_B": "EPB41L2", "ncbiTaxonId": 9606, "score": 0.727, "nscore": 0, "fscore": 0, "pscore": 0, "ascore": 0, "escore": 0, "dscore": 0, "tscore": 0.727}, {"stringId_A": "9606.ENSP00000222374", "stringId_B": "9606.ENSP00000345259", "preferredName_A": "CADM4", "preferredName_B": "EPB41", "ncbiTaxonId": 9606, "score": 0.628, "nscore": 0, "fscore":

In [112]:
# Getting the STRING-API data into a pandas df
string_io = io.StringIO(string_raw_data)
string_df = pd.read_json(string_io)
string_df.head(5)

,stringId_A,stringId_B,preferredName_A,preferredName_B,ncbiTaxonId,score,nscore,fscore,pscore,ascore,escore,dscore,tscore
0,9606.ENSP00000222374,9606.ENSP00000222644,CADM4,MPP6,9606,0.888,0,0,0,0.000,0.045,0,0.888
1,9606.ENSP00000222374,9606.ENSP00000357106,CADM4,CADM3,9606,0.862,0,0,0,0.128,0.000,0,0.849
2,9606.ENSP00000222374,9606.ENSP00000357110,CADM4,EPB41L2,9606,0.727,0,0,0,0.000,0.000,0,0.727
3,9606.ENSP00000222374,9606.ENSP00000345259,CADM4,EPB41,9606,0.628,0,0,0,0.000,0.000,0,0.628
4,9606.ENSP00000222374,9606.ENSP00000264638,CADM4,CNTNAP1,9606,0.610,0,0,0,0.084,0.000,0,0.592


In [113]:
# Cleaning up the string dataframe
string_df_filter_2 = string_df

# Converting the GeneID to UniprotID
string_df_filter_2[["UniprotID_A", "UniprotID_B"]] = string_df_filter_2[["preferredName_A", "preferredName_B"]].map(convert_stringID_to_uniprotName) # takes a long time
string_df_filter_2.head(5)


,stringId_A,stringId_B,preferredName_A,preferredName_B,ncbiTaxonId,score,nscore,fscore,pscore,ascore,escore,dscore,tscore,UniprotID_A,UniprotID_B
0,9606.ENSP00000222374,9606.ENSP00000222644,CADM4,MPP6,9606,0.888,0,0,0,0.000,0.045,0,0.888,CADM4_HUMAN,MPH6_HUMAN
1,9606.ENSP00000222374,9606.ENSP00000357106,CADM4,CADM3,9606,0.862,0,0,0,0.128,0.000,0,0.849,CADM4_HUMAN,CADM3_HUMAN
2,9606.ENSP00000222374,9606.ENSP00000357110,CADM4,EPB41L2,9606,0.727,0,0,0,0.000,0.000,0,0.727,CADM4_HUMAN,E41L2_HUMAN
3,9606.ENSP00000222374,9606.ENSP00000345259,CADM4,EPB41,9606,0.628,0,0,0,0.000,0.000,0,0.628,CADM4_HUMAN,EPB41_HUMAN
4,9606.ENSP00000222374,9606.ENSP00000264638,CADM4,CNTNAP1,9606,0.610,0,0,0,0.084,0.000,0,0.592,CADM4_HUMAN,CNTP1_HUMAN


In [119]:
df_string_pipe = (string_df_filter_2
                  .filter(items = {"UniprotID_A", "UniprotID_B", "score", "escore"}) # filter out the columns that are needed
                  .rename(columns = {"UniprotID_A": "string_interactor_a", 
                                     "UniprotID_B": "string_interactor_b",
                                     "score": "string_score",
                                     "escore": "string_escore"}) # renaming column headers
                  .pipe(get_interactors_for_target, "string_interactor_a", "string_interactor_b", query) # get one column, only the interactor of the query
                  .dropna(subset = ["interactor_of_" + query]) # removing NAs
                  .sort_values(by = "string_escore") # sore of experimental score
                  .drop_duplicates(subset = ["interactor_of_" + query]) # removing duplicates
                  .loc[lambda x: x["string_escore"] != 0] # only getting interactors with an experimental score != 0
                  .drop(columns=["string_interactor_a", "string_interactor_b"])) # drop unwanted columns

df_string_pipe.head(5)


,string_score,string_escore,interactor_of_CADM4_HUMAN
6,0.538,0.045,MPP3_HUMAN
0,0.888,0.045,MPH6_HUMAN
7,0.523,0.049,DLG3_HUMAN
15,0.401,0.113,PRAX_HUMAN
12,0.463,0.464,HECA2_HUMAN


In [114]:
# filter out the columns that are needed
string_df_filter_3 = string_df_filter_2[["UniprotID_A", 
                                        "UniprotID_B", 
                                        "score", 
                                        "escore"]]

# renaming column headers
df_string_filter_3 = string_df_filter_3.rename(columns= {"UniprotID_A": "string_interactor_a", 
                                                        "UniprotID_B": "string_interactor_b",
                                                        "score": "string_score",
                                                        "escore": "string_escore"})
# get one column, only the interactor of the query
df_string_1 = get_interactors_for_target(df_string_filter_3, "string_interactor_a", "string_interactor_b", query)
df_string_1.head(5)
# # removing duplicates but first remove NaN and sort on score
# df_string_1 = df_string_1.dropna(subset = ["interactor_of_" + query])
# df_string_1 = df_string_1.sort_values(by="string_escore")

# # Drop the duplicate rows based on the "id" and "age" columns.
# df_string = df_string_1.drop_duplicates(subset=["interactor_of_" + query])

# # filtering the df on the escore, i.e. filtering out all the interaction score that are equal to zero
# df_string = df_string[df_string["string_escore"] != 0]


# df_string.head(5)
# df_string.shape

,string_interactor_a,string_interactor_b,string_score,string_escore,interactor_of_CADM4_HUMAN
0,CADM4_HUMAN,MPH6_HUMAN,0.888,0.045,MPH6_HUMAN
1,CADM4_HUMAN,CADM3_HUMAN,0.862,0.000,CADM3_HUMAN
2,CADM4_HUMAN,E41L2_HUMAN,0.727,0.000,E41L2_HUMAN
3,CADM4_HUMAN,EPB41_HUMAN,0.628,0.000,EPB41_HUMAN
4,CADM4_HUMAN,CNTP1_HUMAN,0.610,0.000,CNTP1_HUMAN


In [ ]:
# # GETTING THE DATA FROM the HIPPIE-API

# hippie_api_url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/queryHIPPIE.php"

# protein_to_query = "CADM4"
# layer = 1 #to query protein within input set (0) or against all HIPPIE proteins (1, default)
# threshold = 0 #confidence threshold, default is 0
# format = "conc_file" #this generates a tab seperated text file (other interesting input types: "mitab", "browser")

# hippie_request_url = f"{hippie_api_url}?proteins={protein_to_query}&layers={layer}&conf_thres={threshold}&out_type={format}"

# hippie_response = requests.get(hippie_request_url).text

# print(hippie_response)

# print(hippie_request_url)

# # there seems to be an issue with the PHP request response, probably the server is not correctly configured

In [ ]:
# ## !!!!! HIPPIE website seems to be down at the moment, worked but is not reliable, got 500 errors

# # GETTING THE DATA FROM HIPPIE through WEBSCRAPING
# import requests
# import json
# import re
# from bs4 import BeautifulSoup

# protein = query

# url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/query.php?s="+str(protein)

# payload = {}
# headers = {
# "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7", 
# "Accept-Language": "nl-NL,nl;q=0.9,en-US;q=0.8,en;q=0.7,fr;q=0.6" ,
# "Connection": "keep-alive", 
# "Referer": "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/",
# "Upgrade-Insecure-Requests": "1" ,
# "User-Agent": "Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Mobile Safari/537.36" 
# }

# response = requests.request("GET", url, headers=headers, data=payload)
# soup = BeautifulSoup(response.text, "html.parser")
# table = soup.find("tbody") # already skipped the columns names

# # rows = table.find_all("tr")

# # interactions = []
# # for idx, row in enumerate(rows):
# #     data = row.find_all("td")
# #     interaction = {
# #     "Interactor": data[0].text,
# #     "EntrezGeneID": data[1].text,
# #     "GeneSymbol": data[2].text,
# #     "Score": data[3].text
# #     }
# #     interactions.append(interaction) 
   

# print(response)


In [33]:
# GETTING THE DATA FROM THE APID db by WEBSCRAPING

def extract_table(proteinid):
    url = "http://cicblade.dep.usal.es:8080/APID/InteractionsGrid.action?protein1="+str(proteinid)+"&protein2=NA"

    payload = {}
    headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
    "Cookie": "JSESSIONID=086030D12C94A8248DA2B5B9A84C16FA; _ga=GA1.2.581619552.1696441740; _gid=GA1.2.818200239.1696441740; _ga_7JSDHY18SK=GS1.2.1696441740.1.1.1696442421.0.0.0",
    "Referer": "http://cicblade.dep.usal.es:8080/APID/searchProtein.action",
    "Upgrade-Insecure-Requests": "1",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }

    response = requests.request("GET", url, headers=headers, data=payload)
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find("table", id="interactions") #Find the table

    rows = table.find_all("tr") # Find all table rows
    interactions = [] #initialize empty list
    for idx, row in enumerate(rows): #loop over rows, keep index
        if idx == 0: # first row is the header, skip.
            pass
        else:
            try:
                data1 = row.find_all("td") # get column
                interaction = { # build intraction object
                "ProteinA": data1[0].get_text().strip(),
                "ProteinB": data1[1].get_text().strip(),
                "MethodType": data1[2].get_text().strip(),
                "Method": data1[3].get_text().strip(),
                "Publication": re.sub(" +", " ",data1[4].get_text().strip().replace("\n", "")),
                "Source": data1[5].get_text().strip()
                }
                interactions.append(interaction) #append interaction object to result list
            except:
                pass
    return interactions #return the result list


# To find the UniProtID from the protein name
proteinid = convert_uniprotID_uniprotAcNr(query)
apid_results = extract_table(proteinid)
# print(json.dumps(apid_results, indent=4))
print(apid_results)

[{'ProteinA': 'ETBR2_HUMAN', 'ProteinB': 'TAU_HUMAN', 'MethodType': 'indirect', 'Method': 'fluorescence microscopy (MI:0416)', 'Publication': 'Sinsky, J. et al., 2020 (PMID:32357304 )', 'Source': 'IntAct (Acc: EBI-26374340  )'}, {'ProteinA': 'ETBR2_HUMAN', 'ProteinB': 'TAU_HUMAN', 'MethodType': 'indirect', 'Method': 'fluorescence microscopy (MI:0416)', 'Publication': 'Sinsky, J. et al., 2020 (PMID:32357304 )', 'Source': 'IntAct (Acc: EBI-26374346  )'}, {'ProteinA': 'ETBR2_HUMAN', 'ProteinB': 'TAU_HUMAN', 'MethodType': 'indirect', 'Method': 'fluorescence microscopy (MI:0416)', 'Publication': 'Sinsky, J. et al., 2020 (PMID:32357304 )', 'Source': 'IntAct (Acc: EBI-26374350  )'}, {'ProteinA': 'ETBR2_HUMAN', 'ProteinB': 'JAGN1_HUMAN', 'MethodType': 'binary', 'Method': 'two hybrid (MI:0018)', 'Publication': 'null, 0 (PMID:32296183 )', 'Source': 'BioGRID (Acc: 2739025  )'}, {'ProteinA': 'ETBR2_HUMAN', 'ProteinB': 'CYAC3_HUMAN', 'MethodType': 'binary', 'Method': 'two hybrid (MI:0018)', 'Public

In [ ]:
# Getting the data from APID into a df

apid_df = pd.DataFrame(apid_results)

# filter out the columns that are needed
apid_df_filter = apid_df[[  "ProteinA", 
                                "ProteinB", 
                                "Method", 
                                "Publication",
                                "Source"]]

# renaming column headers
df_apid_final = apid_df_filter.rename(columns= {"ProteinA": "apid_interactor_a", 
                                                    "ProteinB": "apid_interactor_b",
                                                    "Method": "apid_method",
                                                    "Publication": "apid_publication",
                                                    "Source": "apid_source"})

# get one column, only the interactor of the query
df_apid_int = get_interactors_for_target(df_apid_final, "apid_interactor_a", "apid_interactor_b", query)

# Now we have some duplicate rows based on the "interactor_of_query" column, so we joining duplicate rows but retaining the information, so converting all the info to one row
df_apid = removeDuplicateRow_butRetainInfo(df_apid_int, "interactor_of_" + query, "apid_method", "apid_publication", "apid_source")

df_apid.shape

In [ ]:
apid_df_pipe = (apid_df
                .filter(items = {"ProteinA", "ProteinB", "Method", "Publication", "Source"})
                .rename(columns = {"ProteinA": "apid_interactor_a", 
                                   "ProteinB": "apid_interactor_b",
                                   "Method": "apid_method",
                                   "Publication": "apid_publication",
                                   "Source": "apid_source"})
                .pipe(get_interactors_for_target, "apid_interactor_a", "apid_interactor_b", query)
                .pipe(removeDuplicateRow_butRetainInfo, "interactor_of_" + query, "apid_method", "apid_publication", "apid_source"))

apid_df_pipe.shape

In [ ]:
# Make an intersecions diagram using the pyUpSet

# Put the data into the right format using a custom function
biogrid_interactors = convert_column_to_list(df_biogrid, "interactor_of_" + query)
IntAct_interactors = convert_column_to_list(intact_df_pipe, "interactor_of_" + query)
string_interactors = convert_column_to_list(df_string_pipe, "interactor_of_" + query)
apid_interactors = convert_column_to_list(apid_df_pipe, "interactor_of_" + query)

# Plot the data into an upSetplot
ppis = from_contents({"BioGrid": biogrid_interactors, "IntAct": IntAct_interactors, "STRING": string_interactors, "APID": apid_interactors})

ax_dict = UpSet(ppis, subset_size="count", show_counts=True).plot()

# saving the upsetplot
plt.savefig("data/interactors_of_" + query + "_intersectionsPlot.png", dpi = 300)

In [ ]:
# merge the separate dataframes together based on one column

dfs = [df_biogrid, intact_df_pipe, df_string_pipe, apid_df_pipe]
final_df = reduce(lambda left, right: pd.merge(left,right, on=["interactor_of_" + query], how="outer"), dfs)

In [ ]:
# filter the dataframe based on the subcellular location, using the Uniprot-API

def get_subcellular_location(protein):
    
    uniprot_api_url = "https://rest.uniprot.org/uniprotkb" 
    format = "json"
    uniprot_request_url = f"{uniprot_api_url}/{protein}?format={format}"
    uniprot_response = requests.get(uniprot_request_url)

    uniprot_json = uniprot_response.json()

    subcellular_locations = []
    for comment in uniprot_json.get("comments", []):
        if comment.get("commentType") == "SUBCELLULAR LOCATION":
            for subcellular_location in comment.get("subcellularLocations", []):
                location_value = subcellular_location.get("location", {}).get("value", "")
                subcellular_locations.append(location_value)
    
    return subcellular_locations
    
# now go over the dataframe and make a new column with the subcellular location
final_df[["subcellularLocation"]] = final_df[["interactor_of_" + query]].map(get_subcellular_location) # takes a long time


In [ ]:
# filter on the following subcellular locations
subcellular_locations = ["Membrane", "membrane",
                         "Cell junction", "cell junction",
                         "Cell projection", "cell projection",
                         "Cell membrane", "cell membrane",
                         "Plasma membrane", "plasma membrane",
                         "Secreted", "secreted",
                         "Extracellular space", "extracellular space",
                         "Extracellular matrix", "extracellular matrix"
                         "Extracellular exosome", "extracellular exosome",
                         "Cell surface", "cell surface"] # for each term I filtered it twice because of capital letters

final_df_filtered = final_df[final_df["subcellularLocation"].apply(lambda x: any(location in subcellular_locations for location in x))]

# undo the list in the "subcellular locations" column
final_df_filtered["subcellularLocation"] = final_df_filtered["subcellularLocation"].apply(lambda x: ", ".join(x))


In [ ]:
pd.options.mode.chained_assignment = None  # default='warn'

final_df_pipe = (final_df_filtered
                 # Convert the human uniprotID to human uniprotAcNr, make new column "interactor_of_VCAM1_uniprotAcNr"
                 .pipe(convert_uniprotID_uniprotAcNr_from_df_column, "interactor_of_" + query, "interactor_of_" + query + "_Uniprot_AcNr")
                 # Convert from human uniprotAcNr to mouseGene MGI
                 .assign(**{"interactor_of_" + query + "_mouseGene_MGI": lambda x: x["interactor_of_" + query + "_Uniprot_AcNr"].map(convert_uniprotAcNr_to_mouseMGI)})
                 # undo the list in the mouseGene MGI column (MGI is necessary for further downstream gene ontology)
                 .assign(**{"interactor_of_" + query + "_mouseGene_MGI": lambda x: x["interactor_of_" + query + "_mouseGene_MGI"].apply(lambda y: ", ".join(y))})
                 # column order
                 .loc[:, ["interactor_of_" + query, "interactor_of_" + query + "_Uniprot_AcNr", "interactor_of_" + query + "_mouseGene_MGI"] 
                      + list(final_df_filtered.columns.difference(["interactor_of_" + query, "interactor_of_" + query + "_Uniprot_AcNr", "interactor_of_" + query + "_mouseGene_MGI"]))])

final_df_pipe.shape

In [ ]:
# For downstream purposes, we also add the uniprot accession numbers, human gene symbols, and the mouse gene ortholog gene symbols

# Convert the human uniprotID to human uniprotAcNr, make new column "interactor_of_VCAM1_uniprotAcNr"
convert1 = convert_uniprotID_uniprotAcNr_from_df_column(final_df_filtered, "interactor_of_" + query, "interactor_of_" + query + "_Uniprot_AcNr")

# Convert from human uniprotAcNr to mouseGene MGI
convert2 = convert1
convert2[["interactor_of_" + query + "_mouseGene_MGI"]] = convert2[["interactor_of_" + query + "_Uniprot_AcNr"]].map(convert_uniprotAcNr_to_mouseMGI)

# undo the list in the mouseGene MGI column (MGI is necessary for further downstream gene ontology)
convert3 = convert2
convert3["interactor_of_" + query + "_mouseGene_MGI"] = convert3["interactor_of_" + query + "_mouseGene_MGI"].apply(lambda x: ", ".join(x))

# column order
column_order = ["interactor_of_" + query, "interactor_of_" + query + "_Uniprot_AcNr", "interactor_of_" + query + "_mouseGene_MGI"]
final_df = convert3[column_order + list(convert3.columns.difference(column_order))]
final_df.head(5)

# exporting to a csv
final_df.to_csv("interactors_of_" + query + ".csv", index = False)

In [ ]:
# In this part I was trying to convert from human_UniprotID -> human_UniprotAcNr -> human_ensembl -> mouse_ensembl -> mouseGeneSymbol
# but I was losing a lot of data with these conversions

# # Convert the human uniprotAcNr to human ensemble gene
# convert2 = convert_uniprotAcNr_to_ensembleGene(convert1, "interactor_of_" + query + "_Uniprot_AcNr", "interactor_of_" + query + "_ensembleGene_human")

# # Convert the human ensemble gene to the mouse ensemble gene ortholog
# convert2[["interactor_of_" + query + "_ensembleGene_mouse"]] = convert2[["interactor_of_" + query + "_ensembleGene_human"]].map(convert_humanEnsemblGene_to_mouseEnsemblGene)

# # drop columns in the "interactor_of_" + query + "_ensembleGene_mouse" column
# convert3 = convert2[convert2["interactor_of_" + query + "_ensembleGene_mouse"] != "NA"] # I"m losing a lot of interactors here !!!!!!!!!
# # Convert from mouse ensembl gene ortholog to mouse gene symbol ortholog
# convert4 = convert_ensemblGene_to_geneSymbol(convert3, "interactor_of_" + query + "_ensembleGene_mouse", "interactor_of_" + query + "_geneSymbol_mouse")

In [ ]:
# The following functions were only used in the development stage

In [ ]:
# this function will convert uniprotName to uniprotID, this function is only used in the development stage

def convert_uniprotName_to_uniprotID(uniprotName):
    
    uniprot_api_url = "https://rest.uniprot.org/uniprotkb/search?query="
    fields = "accession,id"
    format = "tsv"
    
    request_url = f"{uniprot_api_url}{uniprotName}&fields={fields}&format={format}"
    
    uniprot_response = requests.get(request_url)

    if uniprot_response.ok:
        print("Data retrieval through Uniprot API succesfull")
        tsv_data = uniprot_response.text
        df = pd.read_csv(StringIO(tsv_data), sep="\t")
        
        filtered_df = df[df["Entry Name"] == uniprotName]
        uniprotID = filtered_df.iloc[0,0]
    
        return (uniprotID)
    
    else: 
        print("Data retrieval through Uniprot API failed")
    
# example
l = convert_uniprotName_to_uniprotID("ITB1_HUMAN")
print(l)

In [ ]:
# this function will convert uniprotID to uniprotName, this function is only used in the development stage

def convert_uniprotID_to_uniprotName(uniprotID):

    if pd.isna(uniprotID):
        return None
    
    else:
        uniprot_api_url = "https://rest.uniprot.org/uniprotkb/search?query="
        fields = "accession,id"
        format = "tsv"
        request_url = f"{uniprot_api_url}{uniprotID}&fields={fields}&format={format}"
        
        try:
            uniprot_response = requests.get(request_url)
            uniprot_response.raise_for_status()
            
            tsv_data = uniprot_response.text
            df = pd.read_csv(StringIO(tsv_data), sep="\t")
            filtered_df = df[df["Entry"] == uniprotID]
            uniprotName = filtered_df.iloc[0,1]
        
            return (uniprotName)
        
        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
        
        return None

# example
l = convert_uniprotID_to_uniprotName("P05556")
print(l)

In [ ]:
# getting from uniprotAcNr to ensemblGene in batch retrieval using the myGene-API python package

def convert_uniprotAcNr_to_ensembleGene(df, df_column_name, df_new_column_name):
    
    mg = mygene.MyGeneInfo()
    
    # convert column to list
    list_of_column = df[df_column_name].tolist()
    
    # get the data from MyGene-API package
    df_mg = mg.querymany(list_of_column, scopes = "uniprot", fields = "ensembl.gene", as_dataframe = True)
    
    # filter it, convert to list & add column to original df
    df_mg_filtered = df_mg[["ensembl.gene"]]
    list_filtered = df_mg_filtered["ensembl.gene"].tolist()
    
    # append the list to the dataframe
    df[df_new_column_name] = list_filtered    
    
    return df

# test
# dummy dataframe
dummy_data = {"Name":["Karan","Rohit","Sahil","Aryan"],"Proteins":["Q3KPI0","Q96FT7", "Q92478", "O15197"]}
dummy_df = pd.DataFrame(dummy_data)

# test
test = convert_uniprotAcNr_to_ensembleGene(dummy_df, "Proteins", "ensembleGenes")
test.head(5)

In [ ]:
# getting from human ensemblGene to the mouse ortholog ensembleGene using the Ensemble-API, not in batch retrieval

def convert_humanEnsemblGene_to_mouseEnsemblGene(ensembleGeneToConvert):

    ensembl_api_url = "http://rest.ensembl.org/homology/id/"
    origin_species = "human"
    origin_gene = ensembleGeneToConvert
    target_species = "mouse"
    format = "condensed"

    ensembl_request_url = f"{ensembl_api_url}{origin_species}/{origin_gene}?target_species={target_species};format={format};content-type=application/json"

    ensembl_response = requests.get(ensembl_request_url)
    ensembl_json = ensembl_response.json()

    print(ensembl_json)
    
    if len(ensembl_json["data"]) != 0 and len(ensembl_json["data"][0]["homologies"]) != 0:
        ensemble_mouseGene = ensembl_json["data"][0]["homologies"][0]["id"]
    else:
        ensemble_mouseGene = "NA"
    
    return ensemble_mouseGene 

# test
test = convert_humanEnsemblGene_to_mouseEnsemblGene("ENSG00000130167")
print(test)


In [ ]:
# small function to check for duplicates (mostly used in developing stage)

import collections

list1 = [1, 2, 3, 4, 5, 3, 4]

# Check for duplicates
counter = collections.Counter(string_interactors)
if any(count > 1 for count in counter.values()):
    print("The list contains duplicates.")
else:
    print("The list does not contain duplicates.")

# Extract the duplicate values
duplicate_values = [key for key, count in counter.items() if count > 1]
print(duplicate_values)

print(string_interactors)

# STRING has 2 duplicates

In [ ]:
# getting from mouse ensemblGene to the mouse ortholog gene symbol using the Ensemble-API, in batch retrieval

def convert_ensemblGene_to_geneSymbol(df, column_to_convert, new_column_name):

    # converting to list
    list_of_ensemblGenes = df[column_to_convert].tolist()
    
    # retrieving the data through the ensembl-API
    ensembl_api_url = "http://rest.ensembl.org"
    ext = "/lookup/id"
    headers = {"Content-type": "application/json", "Accept": "application/json"}
    ids = {"ids" : list_of_ensemblGenes}

    ensembl_response = requests.post(ensembl_api_url+ext, headers = headers, json = ids)
    ensembl_json = ensembl_response.json()
    
    # creating a list of genes
    geneNames =[]
    for gene_id, gene_info in ensembl_json.items():
        if "display_name" in gene_info:
            geneNames.append(gene_info["display_name"])
        else:
            geneNames.append("NA")
    
    # making a dataframe from key:value pairs so that each ensembl will match the geneSymbol        
    df_json = pd.DataFrame({column_to_convert: list(ensembl_json.keys()), new_column_name: geneNames})
    
    # joining the dataframes by the ensemblID
    df_final = pd.merge(df, df_json, on = column_to_convert, how = "outer")
    
    return df_final


# testing the function
# dummy dataframe
dummy_data = {"Name":["Karan","Rohit","Sahil","Aryan"],"Genes":["ENSG00000156475","ENSG00000107779", "ENSG00000170017", "ENSG00000105767"]}
dummy_df = pd.DataFrame(dummy_data)

# test
test = convert_ensemblGene_to_geneSymbol(dummy_df, "Genes", "GeneSymbol")
test.head(5)